# Step 2 - 단안 깊이 추정 (Monocular Depth Estimation)

**목표**: Depth Anything v2로 Front view의 픽셀별 깊이(`depth_norm`)를 추정해 저장한다. (L_vis는 03에서 BEV 레이캐스팅으로 산출)

**입력**: `output/00_front/{stem}_front.jpg`

**출력**: `output/02_depth/{stem}_depth.npz` (`depth_norm`) + `{stem}_depth_vis.jpg`

**깊이 규약 (중요, 03과 공유)**
- Depth Anything v2 원시 출력은 **disparity**(값이 클수록 가까움).
- 본 노트북은 이를 뒤집어 **`depth_norm`: 0=가깝다, 1=멀다** 로 통일해 저장한다.
- `depth_m = depth_norm * DEPTH_SCALE` 로 근사 미터화(단안은 절대 스케일이 없어 근사값).

## 0. 패키지 설치 (처음 1회)

In [1]:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
# !pip install transformers accelerate pillow opencv-python-headless matplotlib numpy

## 1. 라이브러리 및 경로

In [2]:
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
import torch

IMG_DIR = Path("output/00_front")   # Step 0 Front view
OUT_DIR = Path("output/02_depth")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_PATHS = sorted(IMG_DIR.glob("*_front.jpg"))
print(f"Front view {len(IMG_PATHS)}장 발견")

RESUME = True   # True: 이미 처리된 파일 건너뜀 / False: 전체 재처리

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"디바이스: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Front view 1137장 발견
디바이스: cuda
GPU: NVIDIA GeForce RTX 3070 Ti


## 2. Depth Anything v2 로드 (GPU)

`depth-anything/Depth-Anything-V2-Large-hf` (335M 파라미터, VRAM ~5GB). HF pipeline 사용.

In [3]:
from transformers import pipeline as hf_pipeline

MODEL_ID = "depth-anything/Depth-Anything-V2-Large-hf"
print(f"Depth Anything v2 로드 중: {MODEL_ID}")
depth_pipe = hf_pipeline(task="depth-estimation", model=MODEL_ID,
                         device=0 if device == "cuda" else -1)
print("모델 로드 완료")

c:\Users\Jihyun\Desktop\git\ITS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Depth Anything v2 로드 중: depth-anything/Depth-Anything-V2-Large-hf


Loading weights: 100%|██████████| 503/503 [00:00<00:00, 12882.07it/s]


모델 로드 완료


## 3. 깊이 예측

`predict_depth`는 disparity를 뒤집어 `depth_norm`(0=near, 1=far)으로 반환한다.

In [4]:
DEPTH_SCALE = 50.0   # 근사 최대 가시거리(m). L_vis는 03에서 BEV 레이캐스팅으로 산출.


def predict_depth(img_bgr):
    """BGR -> depth_norm(0=near,1=far), disparity(raw)."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    out = depth_pipe(Image.fromarray(img_rgb))
    disp = np.array(out["depth"]).astype(np.float32)   # disparity: 큰 값 = 가까움
    d_min, d_max = disp.min(), disp.max()
    disp_norm = (disp - d_min) / (d_max - d_min + 1e-8)  # 0~1, 1=가까움
    depth_norm = 1.0 - disp_norm                          # 뒤집기: 0=near, 1=far
    return depth_norm, disp


print("함수 정의 완료")

함수 정의 완료


## 4. 전체 이미지 실행

In [5]:
from tqdm.auto import tqdm

depth_results = {}

# 이미 처리된 파일 로드 (RESUME=True일 때만)
done = set()
if RESUME:
    for p in OUT_DIR.glob("*_depth.npz"):
        stem = p.stem.replace("_depth", "")
        depth_results[stem] = {"depth_norm": None}  # 메모리 절약: 배열은 필요 시 재로드
        done.add(stem)

remaining = [p for p in IMG_PATHS if p.stem not in done]
print(f"이미 완료: {len(done)}장 / 전체: {len(IMG_PATHS)}장 → 남은 작업: {len(remaining)}장")

for img_path in tqdm(remaining, desc="깊이 추정"):
    stem = img_path.stem
    img_bgr = cv2.imread(str(img_path))

    depth_norm, disp = predict_depth(img_bgr)
    depth_results[stem] = {"depth_norm": depth_norm}

    # depth 컬러맵 단독 저장 (03 패널용)
    depth_colored = (cm.plasma(depth_norm)[:, :, :3] * 255).astype(np.uint8)
    depth_colored_bgr = cv2.cvtColor(depth_colored, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(OUT_DIR / f"{stem}_depth_vis.jpg"), depth_colored_bgr)

    # 저장 (depth_norm: 0=near,1=far)
    h, w = img_bgr.shape[:2]
    np.savez_compressed(str(OUT_DIR / f"{stem}_depth.npz"), depth_norm=depth_norm)
    with open(OUT_DIR / f"{stem}_depth_meta.json", "w", encoding="utf-8") as f:
        json.dump({"image": img_path.name, "img_hw": [h, w],
                   "depth_scale": DEPTH_SCALE,
                   "convention": "depth_norm 0=near 1=far"}, f, ensure_ascii=False, indent=2)

print("=== 완료 ===")


깊이 추정: 100%|██████████| 1137/1137 [02:55<00:00,  6.46it/s]

=== 완료 ===


## 5. 요약

In [6]:
print(f"=== 깊이 추정 완료: {len(depth_results)}장 처리됨 ===")
print(f"  저장 위치: {OUT_DIR}")
print("다음 단계: 03_bev_shadow.ipynb")

=== 깊이 추정 요약 ===
  cut_point_1000_pano_4-zgBUPlEx-LFtwRNtIEiQ_fr  depth_norm 저장 완료
  cut_point_1001_pano_IQ4H3AfYxt5vQoSdRa1aUw_fr  depth_norm 저장 완료
  cut_point_1005_pano_sSNXY4Ndp-CnY4O7DZcjPg_fr  depth_norm 저장 완료
  cut_point_1006_pano_Rr8L4nop0nWbNMkf-dzYxg_fr  depth_norm 저장 완료
  cut_point_1007_pano_JCJoTj97-psyPNnNt73bbQ_fr  depth_norm 저장 완료
  cut_point_1008_pano_aHMpHsJEwWH-FjvMTMOsUA_fr  depth_norm 저장 완료
  cut_point_1009_pano_XeD5q-0U1WD2I4FXfRjoAA_fr  depth_norm 저장 완료
  cut_point_100_pano_3h8UuIgEVhDpJILYS2UlFQ_fro  depth_norm 저장 완료
  cut_point_1010_pano_oaDIYeTodpyj-peusZKC0g_fr  depth_norm 저장 완료
  cut_point_1014_pano_4IdLIexntFaB_qQOOWNX0g_fr  depth_norm 저장 완료
  cut_point_1016_pano_KD_eTNUuea9JamXyKe6sCA_fr  depth_norm 저장 완료
  cut_point_101_pano_-leTuFvUyJTvoRgnijTOqQ_fro  depth_norm 저장 완료
  cut_point_1026_pano_tnNrfDc7Hb_L2h825m2A9Q_fr  depth_norm 저장 완료
  cut_point_1028_pano_daqdTzaIMKsuUxJRKXndDA_fr  depth_norm 저장 완료
  cut_point_1029_pano_1BliXBrc9XXRTFqBrQdxUQ_fr  depth_norm